# Industrial Part Scene Understanding
## Blender Synthetic Dataset + Custom Neural Network for Defect Detection

**Course**: MEME552 Deep Learning Theory and Applications, NSYSU

**Topic**: Quality Control & Defect Detection + Sorting & Grasping

---

### Overview

This project builds a complete pipeline for industrial part defect detection:
1. **Blender synthetic dataset** — 4 part types × 4 defect states × multi-view rendering = 2,880 patches
2. **Runtime scene generation** — composites patches onto randomized backgrounds with collision detection
3. **Custom encoder-decoder segmentation network** (DefectSegNet) — dual-head: semantic segmentation + defect heatmap
4. **Interior-ignore supervision** — novel loss weighting that focuses on deformation boundaries

### Key Results (S6 Final)
- **Test defect IoU: 0.443** (vs S3 baseline 0.362, +22%)
- **Test mIoU: 0.732** (vs S3 0.712)
- **Pixel accuracy: 95.02%**

## 1. Environment Setup

In [ ]:
import sys, os
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

# Ensure project root is in path
REPO = Path(".").resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}, device={device}")
print(f"CUDA: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

## 2. Data Source & Preprocessing

### 2.1 Blender Patch Rendering

We use Blender to render 2,880 part patches from 4 CAD models (McMaster-Carr):
- **4 part types**: socket_head, pan_head, hex_nut, flange_nut
- **4 defect states**: normal, bend (45°), displace (surface noise), remesh (topology damage)
- **Multi-view grid**: 3 azimuths × 5 elevations × 4 HDRI environments × 4 rotations × 3 light strengths

Defect modifiers applied via Blender's modifier stack:
- **Bend**: SIMPLE_DEFORM modifier, 45° with ±30° axis jitter
- **Displace**: Subdivision (level 1) + DISPLACE with Clouds texture (strength=0.6, size=0.25)
- **Remesh**: SMOOTH mode, octree_depth=6

In [ ]:
# Show patch statistics
from src.render.config import PARTS, DEFECT_STATES, DEFAULT_RENDER_CONFIG

cfg = DEFAULT_RENDER_CONFIG
n_az = len(cfg.patch_azimuths)
n_el = len(cfg.patch_elevations)
n_hdri = len(cfg.hdris)
n_rot = len(cfg.hdri_rotations)
n_str = len(cfg.hdri_strengths)
n_states = len(DEFECT_STATES)
total = n_az * n_el * n_hdri * n_rot * n_str * n_states

print(f"Parts: {PARTS}")
print(f"Defect states: {DEFECT_STATES}")
print(f"View grid: {n_az} az × {n_el} el = {n_az*n_el} views")
print(f"HDRI variation: {n_hdri} files × {n_rot} rot × {n_str} str = {n_hdri*n_rot*n_str} lighting conditions")
print(f"Total patches: {n_az}×{n_el}×{n_hdri}×{n_rot}×{n_str}×{n_states} = {total}")

### 2.2 Runtime Scene Generation

Training scenes are generated on-the-fly (not stored on disk):
- Place 3-6 parts randomly with collision detection (depth-map based)
- Each part: random scale (50-75% base × ±10% jitter), random rotation, random patch selection
- 20% probability each part is defective
- Domain randomization: real photos, procedural textures, or solid color backgrounds
- Geometric distractors (ellipses, rectangles, polygons) for robustness

In [ ]:
# Generate sample scenes
from src.data import load_assets, generate_scene
from src.data.config import DEFAULT_CONFIG, SAMPLE_INDEX_OFFSET

assets = load_assets()
print(f"Loaded assets: {len(assets.patches)} patch pools, "
      f"{len(assets.backgrounds)} backgrounds")

# Show 4 sample scenes
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    s = generate_scene(SAMPLE_INDEX_OFFSET + i, assets, DEFAULT_CONFIG)
    axes[0, i].imshow(s.rgb)
    axes[0, i].set_title(f"Scene {i} (RGB)", fontsize=9)
    axes[0, i].axis("off")
    
    # Semantic GT: 0=bg(black), 1=normal(green), 2=defect(red)
    cmap = np.array([[0,0,0],[0,200,0],[200,0,0]], dtype=np.uint8)
    axes[1, i].imshow(cmap[s.semantic])
    axes[1, i].set_title("Semantic GT", fontsize=9)
    axes[1, i].axis("off")

fig.suptitle("Sample Generated Scenes (top: RGB, bottom: semantic GT)", fontsize=12)
plt.tight_layout()
plt.show()

### 2.3 Supervision Strategy: Interior-Ignore

The defect head uses a novel supervision approach:
- **Target T**: Gaussian-blurred deformation mask (from Blender normal-vs-defect pixel diff)
- **Weight W**: High at deformation boundaries, ≈0 inside defective parts (interior-ignore)
- This forces the model to learn *boundary* features rather than memorizing interiors
- Normal parts get W=0.3 to learn "no deformation here"

In [ ]:
# Show T/W supervision for a defective scene
from src.data import encode_targets

# Find a scene with defects
for idx in range(50):
    s = generate_scene(SAMPLE_INDEX_OFFSET + 100 + idx, assets, DEFAULT_CONFIG)
    if (s.semantic == 2).sum() > 200:
        break

tg = encode_targets(s.semantic, s.instance, s.meta, s.defect_region)
T = tg["B_T"].numpy()
W = tg["B_W"].numpy()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(s.rgb); axes[0].set_title("RGB")
axes[1].imshow(cmap[s.semantic]); axes[1].set_title("Semantic GT")
axes[2].imshow(T, cmap="hot", vmin=0, vmax=1); axes[2].set_title("Target T (deformation)")
im = axes[3].imshow(W, cmap="viridis", vmin=0, vmax=W.max())
axes[3].set_title("Weight W (interior-ignore)")
plt.colorbar(im, ax=axes[3], fraction=0.046)
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Model Architecture

**DefectSegNet** — a custom U-Net-style encoder-decoder with:
- **Encoder**: 4 levels of ConvBlock (2× Conv3×3 + BN + ReLU) with MaxPool downsampling
- **Decoder**: 4 levels of ConvTranspose2d upsampling + skip connections
- **Dual heads**:
  - Head A (semantic): 3-class segmentation (bg / normal_part / defective_part)
  - Head B (defect): single-channel sigmoid heatmap for deformation localization
- **Parameters**: ~3.3M (base_c=32, depth=4)

This is a **fully custom architecture** — not a direct copy of U-Net, YOLO, or other published models.

In [ ]:
from src.models import DefectSegNet
from src import schema

model = DefectSegNet(base_c=32, depth=4)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: DefectSegNet(base_c=32, depth=4)")
print(f"Parameters: {n_params:,}")
print(f"Heads: {[h.key for h in schema.HEADS]}")
print(f"Output channels: {schema.output_channels()}")

# Test forward pass
dummy = torch.zeros(1, 3, 256, 256)
out = model(dummy)
for k, v in out.items():
    print(f"  {k}: {tuple(v.shape)}")

In [ ]:
# Print model architecture
print(model)

## 4. Training

### Training Configuration
- **Optimizer**: Adam (lr=1e-3, weight_decay=1e-4)
- **Scheduler**: ReduceLROnPlateau (patience=3, factor=0.5, monitoring val defect IoU)
- **Loss**: Cross-entropy (semantic head) + weighted BCE (defect head, pos_weight=8)
- **Steps**: 4000 (batch_size=8, runtime-generated scenes)
- **Best checkpoint**: selected by val defect IoU (not mIoU)

In [ ]:
import json

# Load training history
run_dir = Path("output/runs/s6_final")
history = json.loads((run_dir / "history.json").read_text(encoding="utf-8"))
ckpt = torch.load(run_dir / "best.pt", map_location="cpu", weights_only=False)

print(f"Training config:")
for k, v in ckpt["train_config"].items():
    print(f"  {k}: {v}")
print(f"\nBest val defect IoU: {ckpt.get('best_val_defect_iou', 'N/A')}")
print(f"Best val mIoU: {ckpt['best_val_miou']:.3f}")
print(f"Test metrics: {ckpt['test_metrics']}")

In [ ]:
# Plot training curves
s = history["step"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Loss
axes[0].plot(s, history["L_total"], "-", color="#d4a017", lw=1.5, label="Train")
axes[0].plot(s, history["val_L_total"], "-", color="#8e44ad", lw=1.5, label="Val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Step"); axes[0].legend(); axes[0].grid(alpha=0.3)

# Per-class IoU
axes[1].plot(s, history["val_IoU_bg"], "--", color="#888", lw=1.2, label="bg")
axes[1].plot(s, history["val_IoU_normal"], "--", color="#198754", lw=1.2, label="normal")
axes[1].plot(s, history["val_IoU_defect"], "-o", color="#c0392b", lw=2, markersize=3, label="defect")
axes[1].set_title("Val IoU per Class"); axes[1].set_xlabel("Step")
axes[1].set_ylim(-0.02, 1.02); axes[1].legend(); axes[1].grid(alpha=0.3)

# LR
axes[2].plot(s, history["lr"], "-", color="#198754", lw=2)
axes[2].set_yscale("log")
axes[2].set_title("Learning Rate"); axes[2].set_xlabel("Step"); axes[2].grid(alpha=0.3)

plt.suptitle("S6 Training Dynamics (4000 steps, bc=32, pw=8)", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Results & Prediction

### 5.1 Load Best Model and Run Inference

In [ ]:
from src.models import load_state_dict_flexible
from src.data import encode_rgb
from src.eval.metrics import infer_3class

# Load best model
tc = ckpt["train_config"]
model = DefectSegNet(base_c=tc["base_c"], depth=tc.get("depth", 4)).to(device)
load_state_dict_flexible(model, ckpt["state_dict"])
model.eval()
print(f"Loaded best model: bc={tc['base_c']}, depth={tc.get('depth',4)}")
print(f"Test mIoU: {ckpt['test_metrics']['mIoU']:.3f}")
print(f"Test defect IoU: {ckpt['test_metrics']['IoU_per_class'][2]:.3f}")
print(f"Test pixel acc: {ckpt['test_metrics']['pixel_acc']*100:.2f}%")

In [ ]:
# Generate prediction examples
from src.data.config import TEST_INDEX_OFFSET

cmap3 = np.array([[0,0,0],[0,200,0],[200,0,0]], dtype=np.uint8)

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
col_titles = ["RGB", "GT (3-class)", "Prediction", "P(defect) heatmap"]

shown = 0
for idx in range(200):
    s = generate_scene(TEST_INDEX_OFFSET + idx, assets, DEFAULT_CONFIG)
    has_defect = (s.semantic == 2).sum() > 100
    # Show mix of defective and clean scenes
    if shown < 3 and not has_defect:
        continue
    if shown >= 4:
        break
    
    rgb_t = encode_rgb(s.rgb).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(rgb_t)
        pred, _, pdef = infer_3class(out, tc.get("defect_thr", 0.7))
    
    axes[shown, 0].imshow(s.rgb)
    axes[shown, 1].imshow(cmap3[s.semantic])
    axes[shown, 2].imshow(cmap3[pred[0].cpu().numpy()])
    im = axes[shown, 3].imshow(pdef[0].cpu().numpy(), cmap="jet", vmin=0, vmax=1)
    
    for j in range(4):
        axes[shown, j].axis("off")
        if shown == 0:
            axes[0, j].set_title(col_titles[j], fontsize=11)
    shown += 1

fig.suptitle("Prediction Examples (green=normal, red=defect)", fontsize=13)
plt.tight_layout()
plt.show()

### 5.2 Quantitative Results

In [ ]:
# Final test metrics
tm = ckpt["test_metrics"]
ps = ckpt.get("per_state", {})

print("=" * 50)
print("FINAL TEST METRICS (S6)")
print("=" * 50)
print(f"  Pixel Accuracy: {tm['pixel_acc']*100:.2f}%")
print(f"  mIoU:           {tm['mIoU']:.3f}")
print(f"  Background IoU: {tm['IoU_per_class'][0]:.3f}")
print(f"  Normal IoU:     {tm['IoU_per_class'][1]:.3f}")
print(f"  Defect IoU:     {tm['IoU_per_class'][2]:.3f}")
print()
print("Per-state detection rates:")
for state, info in ps.items():
    print(f"  {state:15s}: det_rate={info['det_rate']*100:5.1f}%  (n_px={info['n_px']})")
print()
print("Comparison with previous stages:")
print(f"  S3 baseline: mIoU=0.712, defect IoU=0.362")
print(f"  S5 best:     mIoU=0.722, defect IoU=0.390")
print(f"  S6 final:    mIoU={tm['mIoU']:.3f}, defect IoU={tm['IoU_per_class'][2]:.3f}  (+{(tm['IoU_per_class'][2]-0.362)/0.362*100:.0f}% over S3)")

In [ ]:
# KPI comparison bar chart
metrics = ["mIoU", "Defect IoU", "Pixel Acc"]
s3 = [0.712, 0.362, 0.941]
s5 = [0.722, 0.390, None]
s6 = [tm["mIoU"], tm["IoU_per_class"][2], tm["pixel_acc"]]

x = np.arange(len(metrics))
w = 0.25
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w, s3, w, label="S3 baseline", color="#9aa0a6")
ax.bar(x, [v if v else 0 for v in s5], w, label="S5", color="#fbbc04")
ax.bar(x + w, s6, w, label="S6 (final)", color="#1a73e8")
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05); ax.set_ylabel("Score")
ax.set_title("Stage Comparison")
ax.legend(); ax.grid(axis="y", alpha=0.3)

for bars in ax.containers:
    for b in bars:
        h = b.get_height()
        if h > 0:
            ax.text(b.get_x() + b.get_width()/2, h + 0.01, f"{h:.3f}", ha="center", fontsize=7)
plt.tight_layout()
plt.show()

## 6. Summary

### Contributions
1. **Custom Blender synthetic dataset** — no public dataset used; full control over defect types and rendering conditions
2. **Runtime scene generation** — collision-aware placement, domain randomization, no data leakage by design
3. **Custom DefectSegNet** — dual-head architecture with interior-ignore supervision
4. **Iterative improvement** across 6 stages: S3→S5→S6, defect IoU improved from 0.362 to 0.443 (+22%)

### Limitations & Future Work
- Bend deformation is difficult to detect (subtle geometric change, near FP noise floor)
- Only pan_head currently has defect patches; extending to all 4 part types would improve generalization
- Model capacity saturates at bc=8; the data bottleneck is defect diversity, not model size